# Experiment: Source audit

Objective:
- State the question you want to answer.
- Define the success criteria.


Inspect the raw snapshot and quantify data-quality issues before filtering.

In [ ]:
from hashlib import sha256

import pandas as pd

from corpus import RAW_PATH

raw = pd.read_csv(RAW_PATH, low_memory=False)
print(f"SHA-256 {sha256(RAW_PATH.read_bytes()).hexdigest()}")
print(f"{len(raw):,} rows and {raw.shape[1]} columns")
raw.dtypes.to_frame("inferred dtype").assign(missing=raw.isna().sum())

In [ ]:
audit = {}
for field in ["id", "budget", "revenue", "runtime"]:
    numeric = pd.to_numeric(raw[field], errors="coerce")
    audit[field] = {
        "invalid non-null": (raw[field].notna() & numeric.isna()).sum(),
        "zeros": numeric.eq(0).sum(),
        "negatives": numeric.lt(0).sum(),
    }

display(pd.DataFrame(audit).T)
display(raw["original_language"].value_counts(dropna=False).head(10))
dates = pd.to_datetime(raw["release_date"], format="%Y-%m-%d", errors="coerce")
print("Unparseable non-null release dates:", (raw["release_date"].notna() & dates.isna()).sum())
print("Rows with repeated raw IDs:", raw["id"].duplicated(keep=False).sum())